# Practice Midterm Design: Student Problems and Solutions (DATA 111, Fall 2026)

These problems were written by your classmates for the Practice Midterm Design assignment. Try each problem **on paper first**, then check the solution below it.

Notes in *italics* marked **Instructor note** point out where a solution needed a correction, or where a problem uses something we haven't covered (and won't be on the midterm).

In [1]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
# Problem 1: Python expressions (ints and floats)

Python treats whole numbers (`int`) and decimals (`float`) differently. What does the last line of this code return? Choose one.

```python
x = 5
y = 2
z = 2.5 * -y
abs(z) - abs(x * y)
```

a. `5.0`  
b. `-5.0`  
c. `5`  
d. `-5`

## Solution 1

**b. `-5.0`.** `z = 2.5 * -2 = -5.0`, a float because 2.5 is a float. Then `abs(z) = 5.0` and `abs(x * y) = abs(10) = 10`, so `5.0 - 10 = -5.0`. The answer is negative, and it is a float because any arithmetic with a float gives a float.

In [2]:
x = 5
y = 2
z = 2.5 * -y
abs(z) - abs(x * y)

-5.0

---
# Problem 2: Table filtering and sorting

A basketball coach collected statistics from six players, including their points scored and minutes played.

Write one line of Python code that finds the **name of the player who scored the most points while playing fewer than 30 minutes**.

In [3]:
players = Table().with_columns(
    'Name', make_array('Jordan', 'Taylor', 'Alex', 'Morgan', 'Casey', 'Riley'),
    'Points', make_array(18, 25, 12, 30, 22, 27),
    'Minutes', make_array(28, 32, 20, 35, 25, 29)
)
players

Name,Points,Minutes
Jordan,18,28
Taylor,25,32
Alex,12,20
Morgan,30,35
Casey,22,25
Riley,27,29


## Solution 2

Keep only the players with fewer than 30 minutes, sort by points (highest first), and take the first name: **Riley** (27 points in 29 minutes). Morgan scored more (30) but played 35 minutes.

In [4]:
players.where('Minutes', are.below(30)).sort('Points', descending=True).column('Name').item(0)

'Riley'

---
# Problem 3: Joining tables

Two surveys asked the same ten people (identified by their age) about their preferences: how they like to get pizza, and how they prefer to pay for groceries.

Using code, find **how many people who prefer to pay for groceries with a credit card also prefer delivery pizza**.

In [5]:
pizza = Table().with_columns(
    'Pizza Prep Preference', make_array('take-n-bake', 'frozen', 'frozen', 'delivery', 'take-n-bake',
                                        'take-n-bake', 'delivery', 'delivery', 'delivery', 'frozen'),
    'Age of Buyer', make_array(18, 27, 42, 39, 26, 31, 44, 19, 20, 35))
pizza

Pizza Prep Preference,Age of Buyer
take-n-bake,18
frozen,27
frozen,42
delivery,39
take-n-bake,26
take-n-bake,31
delivery,44
delivery,19
delivery,20
frozen,35


In [6]:
payment_method = Table().with_columns(
    'Age', make_array(18, 27, 42, 39, 26, 31, 44, 19, 20, 35),
    'Preferred Grocery Payment Method', make_array('credit', 'debit', 'debit', 'cash', 'debit',
                                                   'credit', 'credit', 'cash', 'debit', 'credit'))
payment_method

Age,Preferred Grocery Payment Method
18,credit
27,debit
42,debit
39,cash
26,debit
31,credit
44,credit
19,cash
20,debit
35,credit


## Solution 3

Join the two tables on age so each person's two answers are in the same row, then filter twice and count the rows: **1** person (age 44).

In [7]:
compare_table = pizza.join('Age of Buyer', payment_method, 'Age')
compare_table

Age of Buyer,Pizza Prep Preference,Preferred Grocery Payment Method
18,take-n-bake,credit
19,delivery,cash
20,delivery,debit
26,take-n-bake,debit
27,frozen,debit
31,take-n-bake,credit
35,frozen,credit
39,delivery,cash
42,frozen,debit
44,delivery,credit


In [8]:
compare_table.where('Preferred Grocery Payment Method', 'credit').where('Pizza Prep Preference', 'delivery').num_rows

1

*__Instructor note:__ joining on age works here only because every person has a different age. In real data, two people can share an age, so you would join on an ID column instead.*

---
# Problem 4: Writing a function with table filters

Write a function `apartment_finder(budget, num_people)` that takes a person's monthly budget and how many people are in their group, and returns a table of the apartments that fit: price **at most** the budget and **at least one bed per person**. The cheapest apartments should come first.

In [9]:
apartment_listings = Table().with_columns(
    'Complex Name', make_array('Oakleaf Penthouses', 'College Creek Apartments', 'Evergreen Condos', 'Lakeview Complex',
                               'Windy Hill Living', 'Mainstreet Apartments', 'Benton Duplex'),
    'Price (Monthly)', make_array(10000, 3200, 2500, 2800, 3800, 4200, 4300),
    'Num Beds', make_array(8, 2, 1, 1, 3, 3, 4),
    'Num Baths', make_array(9, 1, 1, 2, 3, 4, 2),
    'Sq Footage', make_array(7500, 1000, 500, 550, 1500, 1550, 1450),
    'Furnished', make_array('Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No'))
apartment_listings

Complex Name,Price (Monthly),Num Beds,Num Baths,Sq Footage,Furnished
Oakleaf Penthouses,10000,8,9,7500,Yes
College Creek Apartments,3200,2,1,1000,Yes
Evergreen Condos,2500,1,1,500,No
Lakeview Complex,2800,1,2,550,Yes
Windy Hill Living,3800,3,3,1500,No
Mainstreet Apartments,4200,3,4,1550,Yes
Benton Duplex,4300,4,2,1450,No


## Solution 4

Filter by price (`are.below_or_equal_to`, because "at most" includes the budget itself), then filter that result by beds (`are.above_or_equal_to`, because "at least"), then sort by price. `sort` is ascending by default, so the cheapest comes first.

In [10]:
def apartment_finder(budget, num_people):
    in_budget = apartment_listings.where('Price (Monthly)', are.below_or_equal_to(budget))
    enough_beds = in_budget.where('Num Beds', are.above_or_equal_to(num_people))
    return enough_beds.sort('Price (Monthly)')

apartment_finder(3000, 1)

Complex Name,Price (Monthly),Num Beds,Num Baths,Sq Footage,Furnished
Evergreen Condos,2500,1,1,500,No
Lakeview Complex,2800,1,2,550,Yes


In [11]:
apartment_finder(3900, 2)

Complex Name,Price (Monthly),Num Beds,Num Baths,Sq Footage,Furnished
College Creek Apartments,3200,2,1,1000,Yes
Windy Hill Living,3800,3,3,1500,No


*__Instructor note:__ using `are.below(budget)` or `are.above(num_people)` would leave out an apartment priced exactly at the budget, or one with exactly one bed per person. Try `apartment_finder(3200, 2)`: College Creek costs exactly 3200 and has exactly 2 beds, so it should be included.*

In [12]:
apartment_finder(3200, 2)

Complex Name,Price (Monthly),Num Beds,Num Baths,Sq Footage,Furnished
College Creek Apartments,3200,2,1,1000,Yes


---
# Problem 5: Reading a function

Here is a function. Answer the questions below it.

*(The author's note: `isinstance(n, int)` on line 2 hasn't been covered in class; read it as "n is an integer".)*

In [13]:
def get_fib(n):
    if not isinstance(n, int):
        return "Number must be a whole number greater than or equal to zero."
    if n < 0:
        return "Number must be a whole number greater than or equal to zero."
    if n == 0:
        return 0
    if n == 1:
        return 1
    a, b = 0, 1
    for i in range(2, n + 1):
        a, b = b, a + b
    return b

1. What does this function do, and what kind of loop does it use?
2. What does its argument need to be?
3. What is the output if the argument is `6`? If it is a negative number? If it is a string?

## Solution 5

1. It returns the number at position `n` in the **Fibonacci sequence** (0, 1, 1, 2, 3, 5, 8, 13, …), where each number is the sum of the two before it. It uses a **`for` loop**.
2. A whole number greater than or equal to 0.
3. `get_fib(6)` returns **8** (positions 0 through 6: 0, 1, 1, 2, 3, 5, **8**). A negative number or a string returns the message `"Number must be a whole number greater than or equal to zero."`

In [14]:
get_fib(6), get_fib(-3), get_fib('6')

(8,
 'Number must be a whole number greater than or equal to zero.',
 'Number must be a whole number greater than or equal to zero.')

*__Instructor note:__ this function uses three things we haven't covered, and none of them will be on the midterm: `isinstance`, `range` (we use `np.arange`, which works the same way here), and `a, b = b, a + b`, which updates two names in one line. Also, the original solution said a boolean would return the error message, but in Python `True` and `False` count as integers (1 and 0), so `get_fib(True)` returns `1`. Try it below.*

In [15]:
get_fib(True), get_fib(False)

(1, 0)

---
# Problem 6: Writing a function with a loop and conditionals

This problem uses the `trails` table from the design template:

In [16]:
trails = Table().with_columns(
    'Name', make_array('Hammond Trail', 'Arcata Marsh Loop', 'Trinidad Head Loop',
                       'Fern Canyon Loop', 'Rhododendron Trail', 'Elk Camp Ridge'),
    'Miles', make_array(5.5, 2.0, 1.4, 1.1, 6.3, 3.8),
    'Dogs', make_array(True, True, True, False, False, True))
trails

Name,Miles,Dogs
Hammond Trail,5.5,True
Arcata Marsh Loop,2,True
Trinidad Head Loop,1.4,True
Fern Canyon Loop,1.1,False
Rhododendron Trail,6.3,False
Elk Camp Ridge,3.8,True


Write a function `find_longer_trail(mileage, dogs_allowed)` that takes a number of miles and a boolean (whether dogs are allowed).

It should return the name of the trail that is **closest in length but longer than** `mileage`, among the trails whose `Dogs` value matches `dogs_allowed`.

If no trail fits, it should print `"No trails for you! Go play Basketball instead."`

## Solution 6

Sort the trails from **shortest to longest**, then go through them in order. The first trail that is longer than `mileage` and has the right `Dogs` value is the closest one, so return it right away. If the loop finishes without returning, no trail fits.

In [17]:
def find_longer_trail(mileage, dogs_allowed):
    by_miles = trails.sort('Miles')          # shortest first
    for i in np.arange(by_miles.num_rows):
        if by_miles.column('Miles').item(i) > mileage:
            if by_miles.column('Dogs').item(i) == dogs_allowed:
                return by_miles.column('Name').item(i)
    print("No trails for you! Go play Basketball instead.")

In [18]:
print(find_longer_trail(0.9, True))    # Trinidad Head Loop (1.4 miles)
print(find_longer_trail(3, True))      # Elk Camp Ridge (3.8 miles)
print(find_longer_trail(1.1, False))   # Rhododendron Trail (6.3 miles)
find_longer_trail(6.4, False)          # no trail is longer than 6.4 miles

Trinidad Head Loop
Elk Camp Ridge
Rhododendron Trail
No trails for you! Go play Basketball instead.


*__Instructor note:__ the original solution sorted the trails from **longest** to shortest, so it returned the longest matching trail instead of the closest one. For example, `find_longer_trail(0.9, True)` returned Hammond Trail (5.5 miles) instead of Trinidad Head Loop (1.4 miles). Good check for your own code: test a case where you know the answer, and make sure the order of your sort matches what the question asks for.*